# 🎙️ VoiceBatch Studio v2.1.6 - [Full Fixed]
यह वर्जन आपके 'Space' एरर और 'FileNotFound' को जड़ से खत्म कर देगा।

In [ ]:
# @title 🔑 Step 1: GitHub Link (Correct Way)
import os, shutil

GITHUB_USER = "" # @param {type:"string"}
GITHUB_TOKEN = "" # @param {type:"string"}
REPO_NAME = "" # @param {type:"string"}

# यूजरनेम को क्लीन करना
CLEAN_USER = GITHUB_USER.strip().replace(" ", "")

if CLEAN_USER and GITHUB_TOKEN and REPO_NAME:
    REPO_URL = f"https://{GITHUB_TOKEN}@github.com/{CLEAN_USER}/{REPO_NAME}.git"
    
    # पुराने कचरे को साफ करना
    if os.path.exists(REPO_NAME): shutil.rmtree(REPO_NAME)
    
    !git clone {REPO_URL}
    
    # फोल्डर के अंदर जाना बहुत जरूरी है
    %cd {REPO_NAME}
    os.makedirs("outputs", exist_ok=True)
    
    !pip install -q gradio librosa soundfile coqui-tts
    print(f"✅ {REPO_NAME} लिंक हो गया! यूजरनेम: {CLEAN_USER}")
else:
    print("⚠️ डेटा भरें!")

In [ ]:
# @title 🚀 Step 2: app.py (No-Stutter Logic)
app_code = r'''
import gradio as gr
import torch
from TTS.api import TTS
import librosa, soundfile as sf
import os, re

device = 'cuda' if torch.cuda.is_available() else 'cpu'
tts = TTS('tts_models/multilingual/multi-dataset/xtts_v2').to(device)

def run_engine(text, audio_sample, speed, pitch, lang, sil_rem):
    # पॉज़ फिक्स
    text = text.replace('...', '. ')
    text = re.sub(r'([।?!,:;])', r' \1 ', text)
    
    out = 'outputs/v_batch_final.wav'
    tts.tts_to_file(text=text, speaker_wav=audio_sample, language=lang, file_path=out, split_sentences=True)
    
    y, sr = librosa.load(out)
    if sil_rem: y, _ = librosa.effects.trim(y, top_db=25)
    if speed != 1.0: y = librosa.effects.time_stretch(y, rate=speed)
    if pitch != 0: y = librosa.effects.pitch_shift(y, sr=sr, n_steps=pitch)
    
    sf.write(out, y, sr)
    return out

with gr.Blocks(theme=gr.themes.Soft(primary_hue="orange")) as demo:
    gr.Markdown('# 🎙️ Master Voice Studio')
    with gr.Row():
        with gr.Column():
            txt = gr.Textbox(label='Script [laugh], [sigh]', lines=6)
            smp = gr.Audio(label='Sample', type='filepath')
            lng = gr.Dropdown(choices=['hi', 'en'], label='Lang', value='hi')
            spd = gr.Slider(0.8, 1.2, 1.0, label='Speed')
            ptc = gr.Slider(-3, 3, 0, label='Pitch')
            btn = gr.Button('Generate 🚀', variant='primary')
        with gr.Column():
            out = gr.Audio(label='Result')
    btn.click(run_engine, [txt, smp, spd, ptc, lng, True], out)
demo.launch(share=True)
'''
with open('app.py', 'w') as f: f.write(app_code)
!python app.py

In [ ]:
# @title ⬆️ Step 3: GitHub Push (No-Space Error Fix)
# कोट्स के साथ यूजरनेम को सुरक्षित करना
!git config --global user.email "user@example.com"
!git config --global user.name "{GITHUB_USER}"
!git add .
!git commit -m "Expression Update"
!git push
print("✅ GitHub पर फाइलें सुरक्षित सेव हो गईं!")